This third notebook solves for the ground-state of the He-like systems, by parameterizing the Slater determinant,
$$
\Psi(\boldsymbol{X}_1,\boldsymbol{X}_2,Z_{\rm{eff}}) =\left[ \psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,1}})\psi_{1\rm{s}}(\boldsymbol{r}_2,Z_{\rm{eff,2}}) +  \psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,1}})\psi_{1\rm{s}}(\boldsymbol{r}_1,Z_{\rm{eff,2}})\right](1+c r_{12}) \left(\frac{1}{\sqrt{2}}\uparrow_1\downarrow_2 -  \frac{1}{\sqrt{2}}\downarrow_1\uparrow_2 \right)
$$

where the bracketed spin term is a normalized Singlet state, and can effectively be ignored for the calculation of Hamiltonian matrix element. $Z_{\rm{eff},1 (2)}$ paramterize each of the one-electron wavefunctions.

The Hamiltonian for a He-like system is,
$$
\mathcal{H} = -\frac{1}{2} \left(\nabla^2_1 + \nabla^2_2 \right) - \frac{Z}{r_1}- \frac{Z}{r_2} + \frac{1}{r_{12}},
$$
where $Z$ is the charge of the nucleus and not to be confused with $Z_{\rm{eff}}$.  

The $r_{12}$ term directly in the above is _possibe_ to do in spherical coordinates (see below attempts) - but better to do it in different coordinates (See notebook 3)

In [26]:
import sympy as sp
sp.init_printing(use_latex='mathjax')
from IPython.display import display
# Define symbolic variables
e, epsilon_0, a = sp.symbols("e epsilon_0 a", positive=True, real=True)
theta2, phi2 = sp.symbols("theta2 phi2", real=True)

psi1 = sp.symbols('psi1')

r1,r2         = sp.symbols('r1 r2',real=True,positive=True)
phi1,phi2     = sp.symbols('phi1 phi2',real=True)
theta1,theta2 = sp.symbols('theta1 theta2',real=True)


u = sp.symbols('u',real=True,positive=True)
Z_eff1     = sp.symbols('Z_eff1',real=True,positive=True)
Z_eff2     = sp.symbols('Z_eff2',real=True,positive=True)
C          = sp.symbols('C',real=True)


Z         = sp.symbols('Z',real=True,positive=True)
E= sp.symbols('E')


psi = sp.Function('psi')



In [27]:
import sympy as sp
from sympy import symbols, Ynm
# 1. Define standard symbols and your function
r, theta, phi = sp.symbols('r theta phi', positive=True)

def attackSQRT(expr):
    #sympy struggles when sqrts are involved. 
    sqrt_map = {
        root: sp.sqrt(root.args[0].factor()) 
        for root in expr.find(sp.Pow) 
        if root.args[1] == sp.Rational(1, 2)
    }
    
    return expr.subs(sqrt_map)

def sphericalLaplacian(f,r,theta,phi):
    term_r = (1 / r**2) * sp.diff(r**2 * sp.diff(f, r), r)
    term_theta = (1 / (r**2 * sp.sin(theta))) * sp.diff(sp.sin(theta) * sp.diff(f, theta), theta)
    term_phi = (1 / (r**2 * sp.sin(theta)**2)) * sp.diff(f, phi, 2)
    laplacian = term_r + term_theta + term_phi
    sp.simplify(laplacian)
    return laplacian


def oneParticleIntegral(expr, r_var, theta_var, phi_var):
    # Assuming expr uses specific variables, or you can substitute them if needed:
    phiint   = sp.Integral(expr, (phi_var, 0, 2*sp.pi)).simplify()
    print('phi   int complete')
    #print(phiint)
    thetaint = sp.Integral(phiint * sp.sin(theta_var), (theta_var, 0, sp.pi)).simplify()
    
    thetaint = thetaint.expand()
    thetaint = attackSQRT(thetaint)
    thetaint
    
    print('theta int complete')

    display(thetaint)
    #The theta integral in the 2-electron integrals introduces sqrts 
    rint     = sp.Integral(thetaint*r_var*r_var, (r_var, 0, sp.oo)).doit()
    
    display(rint)
    rint = rint.simplify()
    
    print('r     int complete')

    #print(rint)
    #print(rint)
    
    return rint

def twoParticleIntegral(expr):

    oneInt = oneParticleIntegral(expr,   r1, theta1, phi1)
    twoInt = oneParticleIntegral(oneInt, r2, theta2, phi2)
    return twoInt



In [28]:
Y = Ynm(1, 0, theta, phi)

In [29]:
Y

Ynm(1, 0, θ, φ)

In [30]:
#sphericalLaplacian(r*Y)

In [31]:
Ynm(0, 0, theta, phi).expand(func=True)**4

  1  
─────
    2
16⋅π 

In [32]:
def oneSorbital(r,theta,phi,Zeff):
    
    return 2 * sp.sqrt(Zeff**3) * sp.exp(-Zeff*r) *  Ynm(0, 0, theta, phi).expand(func=True)

In [33]:
r12 = (sp.sqrt(r1**2 + r2**2 - 2*r1*r2 * sp.cos(theta1)))

In [34]:
oneSorbital(r2,theta2,phi2,Z_eff1)

      3/2  -Z_eff1⋅r₂
Z_eff1   ⋅ℯ          
─────────────────────
         √π          

In [35]:
He_trial_GS = (oneSorbital(r1,theta1,phi1,Z_eff1) * oneSorbital(r2,theta2,phi2,Z_eff2)  + oneSorbital(r1,theta1,phi1,Z_eff2) * oneSorbital(r2,theta2,phi2,Z_eff1)) * (1+C*r12)

In [36]:
He_trial_GS

⎛     _____________________________    ⎞ ⎛      3/2       3/2  -Z_eff1⋅r₂  -Z_ ↪
⎜    ╱   2                       2     ⎟ ⎜Z_eff1   ⋅Z_eff2   ⋅ℯ          ⋅ℯ    ↪
⎝C⋅╲╱  r₁  - 2⋅r₁⋅r₂⋅cos(θ₁) + r₂   + 1⎠⋅⎜──────────────────────────────────── ↪
                                         ⎝                     π               ↪

↪ eff2⋅r₁         3/2       3/2  -Z_eff1⋅r₁  -Z_eff2⋅r₂⎞
↪           Z_eff1   ⋅Z_eff2   ⋅ℯ          ⋅ℯ          ⎟
↪ ─────── + ───────────────────────────────────────────⎟
↪                                π                     ⎠

In [37]:
sp.init_printing(use_latex='mathjax')

ff = He_trial_GS * He_trial_GS
#ff = ff.expand().simplify()
ff

                                        2                                      ↪
⎛     _____________________________    ⎞  ⎛      3/2       3/2  -Z_eff1⋅r₂  -Z ↪
⎜    ╱   2                       2     ⎟  ⎜Z_eff1   ⋅Z_eff2   ⋅ℯ          ⋅ℯ   ↪
⎝C⋅╲╱  r₁  - 2⋅r₁⋅r₂⋅cos(θ₁) + r₂   + 1⎠ ⋅⎜─────────────────────────────────── ↪
                                          ⎝                     π              ↪

↪                                                        2
↪ _eff2⋅r₁         3/2       3/2  -Z_eff1⋅r₁  -Z_eff2⋅r₂⎞ 
↪            Z_eff1   ⋅Z_eff2   ⋅ℯ          ⋅ℯ          ⎟ 
↪ ──────── + ───────────────────────────────────────────⎟ 
↪                                 π                     ⎠ 

In [38]:
sp.init_printing(use_latex='mathjax')


In [39]:
twoParticleIntegral(ff)

phi   int complete
theta int complete


   2       3       3   2  -2⋅Z_eff1⋅r₂  -2⋅Z_eff2⋅r₁      2       3       3    ↪
4⋅C ⋅Z_eff1 ⋅Z_eff2 ⋅r₁ ⋅ℯ            ⋅ℯ               8⋅C ⋅Z_eff1 ⋅Z_eff2 ⋅r₁ ↪
──────────────────────────────────────────────────── + ─────────────────────── ↪
                         π                                                     ↪

↪ 2  -Z_eff1⋅r₁  -Z_eff1⋅r₂  -Z_eff2⋅r₁  -Z_eff2⋅r₂      2       3       3   2 ↪
↪  ⋅ℯ          ⋅ℯ          ⋅ℯ          ⋅ℯ             4⋅C ⋅Z_eff1 ⋅Z_eff2 ⋅r₁  ↪
↪ ───────────────────────────────────────────────── + ──────────────────────── ↪
↪             π                                                                ↪

↪   -2⋅Z_eff1⋅r₁  -2⋅Z_eff2⋅r₂      2       3       3   2  -2⋅Z_eff1⋅r₂  -2⋅Z_ ↪
↪ ⋅ℯ            ⋅ℯ               4⋅C ⋅Z_eff1 ⋅Z_eff2 ⋅r₂ ⋅ℯ            ⋅ℯ      ↪
↪ ──────────────────────────── + ───────────────────────────────────────────── ↪
↪  π                                                      π                    ↪

↪ eff2⋅r₁      2       3 

KeyboardInterrupt: 

In [ ]:
expr = He_trial_GS * He_trial_GS

phiint   = sp.Integral(expr, (phi1, 0, 2*sp.pi)).simplify()
print('phi   int complete')
#print(phiint)
thetaint = sp.Integral(phiint * sp.sin(theta1), (theta1, 0, sp.pi)).simplify()
print('theta int complete')
display(thetaint)
#The theta integral in the 2-electron integrals introduces sqrts 


phi   int complete
theta int complete


                                                                     ⎛         ↪
                                                                   2 ⎜         ↪
        3       3 ⎛ Z_eff1⋅r₁ + Z_eff2⋅r₂    Z_eff1⋅r₂ + Z_eff2⋅r₁⎞  ⎜   2   2 ↪
2⋅Z_eff1 ⋅Z_eff2 ⋅⎝ℯ                      + ℯ                     ⎠ ⋅⎜2⋅C ⋅r₁  ↪
                                                                     ⎝         ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪

↪                         _____________________             __________________ ↪
↪                        ╱   2               2             ╱   2               ↪
↪       2   2   2⋅C⋅r₁⋅╲╱  r₁  - 2⋅r₁⋅r₂ + r₂     2⋅C⋅r₁⋅╲╱  r₁  + 2⋅r₁⋅r₂ + r ↪
↪  + 2⋅C ⋅r₂  - ─────────────────────────────── + ──────────────────────────── ↪
↪                            3⋅r₂                              3⋅r₂            ↪
↪ ─────────────────────────

In [ ]:
thetaint = thetaint.expand()
thetaint = attackSQRT(thetaint).simplify()
thetaint

        3       3 ⎛    2                   7⋅Z_eff1⋅r₁ + 9⋅Z_eff1⋅r₂ + 9⋅Z_eff ↪
4⋅Z_eff1 ⋅Z_eff2 ⋅⎝C⋅r₁ ⋅(r₁ - │r₁ - r₂│)⋅ℯ                                    ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪

↪ 2⋅r₁ + 7⋅Z_eff2⋅r₂         2                   8⋅Z_eff1⋅r₁ + 8⋅Z_eff1⋅r₂ + 8 ↪
↪                    + 2⋅C⋅r₁ ⋅(r₁ - │r₁ - r₂│)⋅ℯ                              ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪ ⋅Z_eff2⋅r₁ + 8⋅Z_eff2⋅r₂       2                   9⋅Z_eff1⋅r₁ + 7⋅Z_eff1⋅r₂ ↪
↪                          + C⋅r₁ ⋅(r₁ - │r₁ - r₂│)⋅ℯ                          ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                              ↪

↪  + 7⋅Z_eff2⋅r₁ + 9⋅Z_ef

In [ ]:
rint     = sp.Integral(thetaint*r1*r1, (r1, 0, sp.oo))
print('r     int complete')

r     int complete


In [ ]:
rint = rint.doit()
rint

KeyboardInterrupt: 

In [ ]:
rint.simplify()

⎛          5       2   2  Z_eff2⋅r₂             5                        5  Z_ ↪
⎝8⋅C⋅Z_eff1 ⋅Z_eff2 ⋅r₂ ⋅ℯ          - 8⋅C⋅Z_eff1 ⋅Z_eff2⋅r₂ + 32⋅C⋅Z_eff1 ⋅ℯ   ↪
────────────────────────────────────────────────────────────────────────────── ↪
                                                                               ↪
                                                                               ↪

↪ eff2⋅r₂              5             2       5   2  Z_eff1⋅r₂                  ↪
↪         - 32⋅C⋅Z_eff1  + 8⋅C⋅Z_eff1 ⋅Z_eff2 ⋅r₂ ⋅ℯ          - 8⋅C⋅Z_eff1⋅Z_e ↪
↪ ──────────────────────────────────────────────────────────────────────────── ↪
↪                                                                 7/2       7/ ↪
↪                                                           Z_eff1   ⋅Z_eff2   ↪

↪    5                 5  Z_eff1⋅r₂              5           5       2     Z_e ↪
↪ ff2 ⋅r₂ + 32⋅C⋅Z_eff2 ⋅ℯ          - 32⋅C⋅Z_eff2  + 8⋅Z_eff1 ⋅Z_eff2 ⋅r₂⋅ℯ    ↪
↪ ────────────────────────